# Loan Assessment System 
In this project, we build a dual-model Machine Learning system. 
1. **Approval Classifier:** Predicts the probability of a loan being approved based on financial history.
2. **Amount Regressor:** Predicts the estimated loan amount a customer is likely to request.

We will use scikit-learn `Pipelines` with `ColumnTransformer` to ensure robust preprocessing. 

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn modules
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, r2_score, mean_absolute_error

# Set plot style
sns.set_theme(style="whitegrid")

## 1. Data Loading and Basic Cleaning
We load the dataset and strip any leading or trailing whitespaces from the column names and categorical values.

In [2]:
# Load the dataset
df = pd.read_csv('loan_approval_dataset.csv')

# Clean column names
df.columns = df.columns.str.strip()

# Clean string values
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].str.strip()

display(df.head())

,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


## 2. Global Preprocessing Setup
We create a universal `ColumnTransformer` to handle both categorical features (using `OneHotEncoder`) and numerical features (using `StandardScaler`).

In [3]:
# Define features (X). We drop the ID and the two target variables.
X = df.drop(columns=['loan_id', 'loan_status', 'loan_amount'])

# Identify column types
categorical_cols = ['education', 'self_employed']
numeric_cols = [col for col in X.columns if col not in categorical_cols]

# Build the preprocessor
preprocessor = make_column_transformer(
    (OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols),
    (StandardScaler(), numeric_cols),
    remainder='passthrough'
)

print("Preprocessor built successfully.")

Preprocessor built successfully.


## 3. Model 1: Loan Approval Probability (Classification)
We will train a `RandomForestClassifier` within a pipeline. 
We split the data taking the first 80% for training and the last 20% for testing (`shuffle=False`).

In [4]:
# Encode the categorical target variable (Approved = 1, Rejected = 0)
y_class = np.where(df['loan_status'] == 'Approved', 1, 0)

# Sequential Train-test split (80% Train, 20% Test) without random shuffling
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X, y_class, test_size=0.2, shuffle=False)

# Create and train the Classification Pipeline
clf_pipeline = make_pipeline(preprocessor, RandomForestClassifier(n_estimators=100, random_state=42))
clf_pipeline.fit(X_train_c, y_train_c)

# Evaluate
y_pred_c = clf_pipeline.predict(X_test_c)
print("--- Classification Model Performance (Sequential Split) ---")
print(f"Accuracy: {accuracy_score(y_test_c, y_pred_c) * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test_c, y_pred_c, target_names=['Rejected', 'Approved']))

--- Classification Model Performance (Sequential Split) ---
Accuracy: 96.25%

Classification Report:
               precision    recall  f1-score   support

    Rejected       0.94      0.95      0.95       299
    Approved       0.97      0.97      0.97       555

    accuracy                           0.96       854
   macro avg       0.96      0.96      0.96       854
weighted avg       0.96      0.96      0.96       854



## 4. Model 2: Loan Amount Prediction (Regression)
Using the same sequential split, we will train a `Ridge` Regression model to predict the continuous target `loan_amount`.

In [5]:
# Define target variable for regression
y_reg = df['loan_amount']

# Sequential Train-test split (80% Train, 20% Test) without random shuffling
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y_reg, test_size=0.2, shuffle=False)

# Create and train the Regression Pipeline
reg_pipeline = make_pipeline(preprocessor, Ridge(alpha=1.0))
reg_pipeline.fit(X_train_r, y_train_r)

# Evaluate
y_pred_r = reg_pipeline.predict(X_test_r)
print("--- Regression Model Performance (Sequential Split) ---")
print(f"R2 Score: {r2_score(y_test_r, y_pred_r):.4f}")
print(f"Mean Absolute Error (MAE): ${mean_absolute_error(y_test_r, y_pred_r):,.2f}")

--- Regression Model Performance (Sequential Split) ---
R2 Score: 0.8671
Mean Absolute Error (MAE): $2,491,398.13


## 5. System Integration: Assessing a New Customer
Let's simulate a scenario where a new customer applies for a loan. Our system will output both the predicted amount they might ask for and the probability of their loan being approved.

In [6]:
# Simulate a new customer profile
new_customer = pd.DataFrame({
    'no_of_dependents': [2],
    'education': ['Graduate'],
    'self_employed': ['No'],
    'income_annum': [8500000],
    'loan_term': [15],
    'cibil_score': [750], # Good credit score
    'residential_assets_value': [15000000],
    'commercial_assets_value': [5000000],
    'luxury_assets_value': [12000000],
    'bank_asset_value': [6000000]
})

# 1. Predict the likely loan amount using the Ridge Regression Pipeline
predicted_amount = reg_pipeline.predict(new_customer)[0]

# 2. Predict the approval probability using the Random Forest Pipeline
# predict_proba returns [[prob_Class_0, prob_Class_1]]
approval_probability = clf_pipeline.predict_proba(new_customer)[0][1] * 100 

print("CUSTOMER ASSESSMENT REPORT")
print(f"Predicted Loan Amount Request: ${predicted_amount:,.2f}")
print(f"Probability of Approval:       {approval_probability:.2f}%")

if approval_probability > 70:
    print("Recommendation:            🟢 HIGHLY LIKELY TO BE APPROVED")
elif approval_probability > 40:
    print("Recommendation:            🟡 MANUAL REVIEW REQUIRED")
else:
    print("Recommendation:            🔴 LIKELY TO BE REJECTED")

CUSTOMER ASSESSMENT REPORT
Predicted Loan Amount Request: $25,446,356.21
Probability of Approval:       98.00%
Recommendation:            🟢 HIGHLY LIKELY TO BE APPROVED
